<div dir="rtl">
<h1>Batch آخر نباید امتیاز اضافی بگیرد</h1>
<p>درس 54 از 76 · ارزیابی دقیقاً چه چیزی را میانگین می‌گیرد؟ · <code dir="ltr">48-evaluate</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/48-evaluate.html">📖 بازگشت به همین درس</a></p>
<p>میانگین را بر تعداد Targetها وزن دهید و آن را با evaluate واقعی مقایسه کنید.</p><p>پیش‌نیاز: Loss میانگین هر Batch و حالت eval/no_grad.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>میانگین خطاهای ۱ و ۳، با تعداد هدف‌های ۸ و ۲، باید به کدام مقدار نزدیک‌تر باشد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from mini_gpt.dataset import NextTokenDataset
from mini_gpt.evaluate import evaluate
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.2))
data = NextTokenDataset([1,2,3,4,5,6,7,8],3)
print('windows:', len(data), 'real evaluation:', evaluate(model, DataLoader(data,batch_size=2), 'cpu'))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>weighted_mean(losses, counts) میانگین Loss هر Batch و تعداد Targetهایش را می‌گیرد و میانگین کل را برمی‌گرداند. ورودی تمرین فهرست‌های هم‌اندازه و ناتهی با counts مثبت است.</p>
</div>

In [ ]:
def weighted_mean(losses, counts):
    # TODO: هر Target سهم برابر دارد
    return None

In [ ]:
def test_exercise():
    result = weighted_mean([1.0,3.0], [8,2])
    if result is None:
        return False
    assert math.isclose(result,1.4)
    assert math.isclose(weighted_mean([1.0,3.0],[2,2]),2.0)
    assert math.isclose(weighted_mean([2.5],[7]),2.5)
    losses, counts = [], []
    model.eval()
    with torch.no_grad():
        for inputs, targets in DataLoader(data,batch_size=2):
            losses.append(model(inputs,targets)[1].item())
            counts.append(targets.numel())
    assert math.isclose(weighted_mean(losses,counts), evaluate(model,DataLoader(data,batch_size=2),'cpu'), abs_tol=1e-7)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: weighted_mean')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط batch_size ارزیابی را عوض کنید. چرا Loss کل باید تقریباً ثابت بماند؟</p>
</div>

In [ ]:
before = {name:value.clone() for name,value in model.state_dict().items()}
for size in (1,2,3):
    print(size, evaluate(model,DataLoader(data,batch_size=size),'cpu'))
assert all(torch.equal(before[name],value) for name,value in model.state_dict().items())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>میانگین Perplexity Batch‌ها با exp میانگین Loss یکی نیست. perplexity_from_batches(losses, counts) را با میانگینِ وزن‌دار Loss تعمیر کنید.</p>
</div>

In [ ]:
wrong = (8*math.exp(1.0)+2*math.exp(3.0))/10
print('wrong mean of perplexities:', wrong)
print('exp of mean loss:', math.exp(1.4))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def perplexity_from_batches(losses, counts):
    # TODO: ابتدا Loss کل، سپس exp
    return None

In [ ]:
def test_repair():
    result = perplexity_from_batches([1.0,3.0],[8,2])
    if result is None:
        return False
    assert math.isclose(result, math.exp(1.4))
    assert math.isclose(perplexity_from_batches([0.0,0.0],[1,4]),1.0)
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: perplexity_from_batches')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>evaluate پروژه دقیقاً از targets.numel() برای وزن‌دهی استفاده می‌کند. پنجره‌ها هم‌پوشان‌اند؛ این عدد را با سنجش تک‌گذری یک متن دیگر یکی ندانید.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>برابری نتیجه با چند batch_size کدام خطای پیاده‌سازی را آشکار می‌کند و چه چیزی دربارهٔ کیفیت زبان نمی‌گوید؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/48-evaluate.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/48-evaluate.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>